# Evaluation Results
This notebook loads evaluation CSVs from `RELDEC/notebook_runs/continuous_reldec/active_run/*/results` and plots FER vs SNR for each method.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import seaborn as sns
sns.set(style='whitegrid')

def load_accumulated(run_name):
    base = Path('RELDEC/notebook_runs/continuous_reldec/active_run') / run_name / 'results'
    acc = base / 'accumulated_results.csv'
    if acc.exists():
        return pd.read_csv(acc)
    frames = []
    if base.exists():
        for p in sorted(base.glob('eval_*.csv')):
            try:
                frames.append(pd.read_csv(p))
            except Exception as e:
                print('Failed reading', p, e)
    if frames:
        return pd.concat(frames, ignore_index=True)
    return pd.DataFrame()

In [ ]:
def plot_run(run_name):
    df = load_accumulated(run_name)
    if df.empty:
        print('No data for', run_name)
        return
    df['snr_db'] = df['snr_db'].astype(float)
    if 'fer' in df.columns:
        df['fer'] = df['fer'].astype(float)
    else:
        df['fer'] = df['frame_errors'] / df['frames']
    plt.figure(figsize=(6,4))
    for method, g in df.groupby('method'):
        g = g.sort_values('snr_db')
        plt.plot(g['snr_db'], g['fer'], marker='o', label=method)
    plt.xlabel('SNR (dB)')
    plt.ylabel('FER')
    plt.title(run_name)
    plt.legend()
    plt.show()

# Plot both runs
plot_run('mackay')
plot_run('ab')